# `slimtsf`: Quickstart Tutorial
### Sliding-Window Multivariate Time-Series Forest with Bootstrap Feature Stability Selection

This tutorial demonstrates:
1. Generating a synthetic 3-channel sensor / motion time-series dataset.
2. Training `SlimTSFClassifier` with multi-scale intervals and bootstrap feature stability selection.
3. Evaluating classification metrics and interpreting top selected features.
4. Using modular feature transformers (`SlidingWindowIntervalTransformer` & `IntervalStatsPoolingTransformer`) independently.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

from slimtsf import (
    SlimTSFClassifier,
    SlidingWindowIntervalTransformer,
    IntervalStatsPoolingTransformer,
)

## 1. Simulate Tri-Axial Sensor Data
We simulate a 3-channel dataset representing motion across three classes: **Resting (0)**, **Walking (1)**, and **Running (2)**.

In [ ]:
rng = np.random.default_rng(42)
n_samples_per_class = 40
n_channels = 3
n_timepoints = 120
total_samples = n_samples_per_class * 3

X = rng.standard_normal((total_samples, n_channels, n_timepoints)) * 0.5
y = np.repeat(np.arange(3), n_samples_per_class)
t = np.linspace(0, 4 * np.pi, n_timepoints)

# Class 1: Walking rhythm
for i in np.where(y == 1)[0]:
    X[i, 0, :] += np.sin(t) * 2.0
    X[i, 1, :] += np.cos(t) * 1.0

# Class 2: Running high-frequency acceleration
for i in np.where(y == 2)[0]:
    X[i, 0, :] += np.sin(2.5 * t) * 3.5
    X[i, 1, :] += np.cos(2.5 * t) * 3.0
    X[i, 2, 60:] += 2.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape},  y_test shape:  {y_test.shape}")

## 2. Train `SlimTSFClassifier`
We train the classifier using multi-scale intervals (`[16, 32, 64]`), summary statistics, and multi-pass bootstrap stability selection (`importance_method="gini"`).

In [ ]:
clf = SlimTSFClassifier(
    window_sizes=[16, 32, 64],
    window_step_ratio=0.5,
    feature_functions=["mean", "std", "slope"],
    aggregations=("min", "mean", "max"),
    feature_mode="both",
    bootstrap=True,
    bootstrap_run=10,
    top_rank=5,
    importance_method="gini",
    n_estimators=100,
    random_state=42,
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=["Resting", "Walking", "Running"]))

## 3. Inspect Selected Stable Features

In [ ]:
print("Top Selected Features across Bootstrap Passes:")
for rank, (name, count) in enumerate(clf.get_feature_selection_frequencies()[:5], start=1):
    print(f"{rank}. {name} (selected in {count}/{clf.bootstrap_run} passes)")

## 4. Modular Transformers Usage
You can also use the feature extraction stages independently within scikit-learn pipelines.

In [ ]:
stage1 = SlidingWindowIntervalTransformer(window_sizes=[16, 32])
feat_intervals = stage1.fit_transform(X_train)

stage2 = IntervalStatsPoolingTransformer(aggregations=("min", "mean", "max"))
feat_pooled = stage2.fit_transform(feat_intervals, feature_metadata=stage1.feature_metadata_)

print(f"Interval features shape: {feat_intervals.shape}")
print(f"Pooled features shape:   {feat_pooled.shape}")